In [ ]:
!pip install opencv-python numpy scipy face_recognition fer moviepy SpeechRecognition tqdm mediapipe insightface
!pip uninstall -y onnxruntime onnxruntime-gpu mediapipe protobuf
!pip install -q opencv-python tensorflow deepface scipy insightface ultralytics mediapipe
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 32.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of facenet-pytorch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.1/891.1 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 109.0 MB/s eta 0:00:00
  Created wheel for insightface:

In [ ]:

import cv2
import numpy as np
import insightface
from ultralytics import YOLO
from deepface import DeepFace
from scipy.spatial import distance as dist
from collections import deque, Counter
import math
import os
import mediapipe as mp
import insightface
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab import drive
from insightface.app import FaceAnalysis
from numpy.linalg import norm

BASE_DRIVE = '/content/drive'
drive.mount(BASE_DRIVE)

PATH_TECH_04 = f'{BASE_DRIVE}/MyDrive/Colab Notebooks/tech 04'
MODEL_PATH = 'hand_landmarker.task'

CHUNK = ''
INPUT_VIDEO = os.path.join(PATH_TECH_04, 'video', f'input_video{CHUNK}.mp4')
OUTPUT_VIDEO = os.path.join(PATH_TECH_04, f'output_analysis{CHUNK}.mp4')
OUTPUT_AUDIO = os.path.join(PATH_TECH_04, f'extracted_audio{CHUNK}.wav')
OUTPUT_TEXT = os.path.join(PATH_TECH_04, f'transcription{CHUNK}.txt')
KNOWN_FACES_DIR = os.path.join(PATH_TECH_04, 'known_faces')

if not os.path.exists(INPUT_VIDEO):
    raise ValueError(f"❌ ARQUIVO NÃO ENCONTRADO: {INPUT_VIDEO}")

# Parâmetros
LOG_FRAMES = 50
MOVEMENT_THRESHOLD = 5.0
# Apertei a distância do MP: 0.08 (8% da tela) para exigir que estejam se tocando
HANDSHAKE_DIST_MP = 0.08
HANDSHAKE_DIST_YOLO = 60.0
EMOTION_WINDOW = 5

global_stats = {'frames': 0, 'anomalies': 0, 'emotions': [], 'activities': []}

face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

try:
    face_app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
except:
    face_app = insightface.app.FaceAnalysis(name='buffalo_l')
face_app.prepare(ctx_id=0, det_size=(640, 640))

Mounted at /content/drive
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 75163.50KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Applied prov

In [ ]:
def load_know_faces(folder):
  know_embeddings = []
  know_names = []

  for file in os.listdir(folder):
    print(f'Carregando rosto - {file}')
    if file.lower().endswith(('.jpg', '.png', '.jpeg')):
      path = os.path.join(folder,file)
      img = cv2.imread(path)
      img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
      faces = face_app.get(img_rgb)

      if not faces:
        print(f'Nenhum rosto em {file}')
        continue

      emb = faces[0].embedding
      name = os.path.splitext(file)[0].rsplit('_', 1)[0]
      know_embeddings.append(emb)
      know_names.append(name)
      print(f'Rosto carregado com sucesso - {file}')

  return know_embeddings, know_names

know_embeddings, know_names = load_know_faces(KNOWN_FACES_DIR)
print("✅ Embeddings carregados:", len(know_embeddings))

def match_face(embedding, threshold=0.45):

    if embedding is None or not know_embeddings:
        return 'Desconhecido'

    similarities = [
        np.dot(embedding, known_emb) /
        (np.linalg.norm(embedding) * np.linalg.norm(known_emb))
        for known_emb in know_embeddings
    ]

    best_idx = int(np.argmax(similarities))
    best_score = similarities[best_idx]

    return know_names[best_idx] if best_score >= threshold else 'Desconhecido'

Carregando rosto - Joao.png
Rosto carregado com sucesso - Joao.png
Carregando rosto - Maria.png
Rosto carregado com sucesso - Maria.png
Carregando rosto - Ricardo.png
Rosto carregado com sucesso - Ricardo.png
Carregando rosto - Luciano.png
Rosto carregado com sucesso - Luciano.png
Carregando rosto - Cristian.png
Rosto carregado com sucesso - Cristian.png
Carregando rosto - Sabrina.png
Rosto carregado com sucesso - Sabrina.png
Carregando rosto - Maria_2.png
Rosto carregado com sucesso - Maria_2.png
Carregando rosto - Thomas.png
Rosto carregado com sucesso - Thomas.png
Carregando rosto - Solange.png
Rosto carregado com sucesso - Solange.png
Carregando rosto - Dr Martin.png
Rosto carregado com sucesso - Dr Martin.png
Carregando rosto - Paulo.png
Nenhum rosto em Paulo.png
Carregando rosto - Dr Martin_01.png
Rosto carregado com sucesso - Dr Martin_01.png
Carregando rosto - Paciente.png
Rosto carregado com sucesso - Paciente.png
Carregando rosto - Cristiano.png
Rosto carregado com sucesso - 

In [ ]:

# --- 3. MODELOS ---
print("\n⚙️ Inicializando Modelos...")
pose_model = YOLO('yolov8n-pose.pt')


global_stats['frames'] = 0

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2, # Foca em apenas duas mãos no close-up
    min_hand_detection_confidence=0.6, # Aumentei a confiança necessária
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6)
detector = vision.HandLandmarker.create_from_options(options)
print("✅ Tudo pronto.")

# --- FUNÇÕES ---
# (Funções auxiliares mantidas da V17 para rotação, emoção e câmera)
def get_camera_shift(trackers):
    deltas_x, deltas_y = [], []
    for tid, data in trackers.items():
        if data['prev_kpts'] is not None and data['kpts_yolo'] is not None:
            valid = (data['kpts_yolo'][[0, 5, 6], 2] > 0.5)
            if np.sum(valid) >= 2:
                diff = np.mean(data['kpts_yolo'][[0, 5, 6], :2][valid] - data['prev_kpts'][[0, 5, 6], :2][valid], axis=0)
                deltas_x.append(diff[0]); deltas_y.append(diff[1])
    if len(deltas_x) > 0: return np.median(deltas_x), np.median(deltas_y)
    return 0.0, 0.0

def get_face_rotation(landmarks):
    if landmarks is None: return 0
    eye_l, eye_r = landmarks[0], landmarks[1]
    dy, dx = eye_r[1] - eye_l[1], eye_r[0] - eye_l[0]
    return abs(np.degrees(np.arctan2(dy, dx)))

def clean_emotion(raw, conf, angle):
    if angle > 30.0: return "neutral"
    if conf < 60: return "neutral"
    if raw in ['angry', 'fear'] and conf < 95: return "focused"
    if raw == 'sad' and conf < 80: return "neutral"
    return raw

def is_valid_handshake_vector(kpts1, kpts2):
    if kpts1[8][2] < 0.5 or kpts1[10][2] < 0.5 or kpts2[8][2] < 0.5 or kpts2[10][2] < 0.5: return False
    elbow1, wrist1 = kpts1[8][:2], kpts1[10][:2]
    elbow2, wrist2 = kpts2[8][:2], kpts2[10][:2]
    if np.linalg.norm(wrist1 - wrist2) > HANDSHAKE_DIST_YOLO: return False
    vec1, vec2 = wrist1 - elbow1, wrist2 - elbow2
    norm1, norm2 = np.linalg.norm(vec1), np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0: return False
    if np.dot(vec1/norm1, vec2/norm2) < -0.5: return True
    return False

def analyze_motion_compensated(curr_kpts, prev_kpts, cam_dx, cam_dy):
    nose = curr_kpts[0]
    l_wr, r_wr = curr_kpts[9], curr_kpts[10]
    l_sh, r_sh = curr_kpts[5], curr_kpts[6]
    activity, is_anomaly = "Parado", False

    arms_up = False
    if l_wr[2]>0.5 and l_sh[2]>0.5 and l_wr[1]<l_sh[1]: arms_up=True
    if r_wr[2]>0.5 and r_sh[2]>0.5 and r_wr[1]<r_sh[1]: arms_up=True
    if arms_up: return "Bracos Levantados", False

    if nose[2]>0.5 and ((l_wr[2]>0.5 and np.linalg.norm(l_wr[:2]-nose[:2])<80) or
                        (r_wr[2]>0.5 and np.linalg.norm(r_wr[:2]-nose[:2])<80)):
        return "Mao no Rosto", False

    if prev_kpts is not None:
        curr_torso_x = (l_sh[0] + r_sh[0]) / 2
        curr_torso_y = (l_sh[1] + r_sh[1]) / 2
        prev_torso_x = (prev_kpts[5][0] + prev_kpts[6][0]) / 2
        prev_torso_y = (prev_kpts[5][1] + prev_kpts[6][1]) / 2

        real_move_x = (curr_torso_x - prev_torso_x) - cam_dx
        real_move_y = (curr_torso_y - prev_torso_y) - cam_dy
        real_move_mag = math.hypot(real_move_x, real_move_y)

        if real_move_mag > 100: pass
        elif real_move_mag > 15.0: activity, is_anomaly = "ANOMALIA (Brusco)", True
        elif real_move_mag > MOVEMENT_THRESHOLD: activity = "Movimento"

    return activity, is_anomaly

def generate_report():
    print("\n" + "="*50); print("📊 RELATÓRIO FINAL V18"); print("="*50)
    print(f"Frames: {global_stats['frames']} | Anomalias: {global_stats['anomalies']}")
    print("-" * 50); print("ATIVIDADES:", [x for x in Counter(global_stats['activities']).most_common() if x[0] != "Parado"])
    print("="*50)

# --- LOOP PRINCIPAL ---
cap = cv2.VideoCapture(INPUT_VIDEO)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

trackers = {}
next_id = 0

print(f"▶️ Processando: {INPUT_VIDEO}")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    global_stats['frames'] += 1
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 1. DETECÇÕES
    yolo_out = pose_model(frame, verbose=False)[0]
    skeletons = []
    if yolo_out.keypoints is not None:
        data = yolo_out.keypoints.data.cpu().numpy()
        for k in data:
            if np.mean(k[:, 2]) > 0.4:
                skeletons.append({'kpts': k, 'nose': k[0][:2]})

    faces = face_app.get(frame_rgb)
    current_faces = []
    for face in faces:
        x1, y1, x2, y2 = face.bbox.astype(int)
        name = match_face(face.embedding)
        current_faces.append({
            'bbox': (x1, y1, x2, y2),
            'centroid': ((x1 + x2) // 2, (y1 + y2) // 2),
            'kps': face.k,
            'name': name
        })

    # 2. DETECÇÃO DE MÃOS COM MEDIAPIPE (SEMPRE ATIVA)
    mp_handshake_box = None
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    detection_result = detector.detect(mp_image)
    if len(detection_result.hand_landmarks) >= 2:
        wrist_pts = []
        for hand_lms in detection_result.hand_landmarks:
            wrist = hand_lms[0]
            wrist_pts.append((wrist.x * w, wrist.y * h))

        for i in range(len(wrist_pts)):
            for j in range(i + 1, len(wrist_pts)):
                if math.hypot(wrist_pts[i][0] - wrist_pts[j][0], wrist_pts[i][1] - wrist_pts[j][1]) < (w * HANDSHAKE_DIST_MP):
                    cx, cy = int((wrist_pts[i][0] + wrist_pts[j][0]) / 2), int((wrist_pts[i][1] + wrist_pts[j][1]) / 2)
                    mp_handshake_box = (cx, cy)
                    global_stats['activities'].append("Aperto de Mao (Close-up)")

    # 3. TRACKING
    active_ids = []
    if len(trackers) > 0 and len(current_faces) > 0:
        t_ids = list(trackers.keys())
        D = dist.cdist([trackers[t]['centroid'] for t in t_ids], [f['centroid'] for f in current_faces])
        rows = D.min(axis=1).argsort()
        cols = D.argmin(axis=1)[rows]
        used = set()
        for r, c in zip(rows, cols):
            if c in used or D[r, c] > 200:
                continue
            tid = t_ids[r]
            trackers[tid].update({
                'centroid': current_faces[c]['centroid'],
                'bbox': current_faces[c]['bbox'],
                'kps': current_faces[c]['kps'],
                'name': current_faces[c]['name'],
                'missing': 0
            })
            active_ids.append(tid)
            used.add(c)

    for i, f in enumerate(current_faces):
        exists = any(np.linalg.norm(np.array(trackers[tid]['centroid']) - np.array(f['centroid'])) < 30 for tid in active_ids)
        if not exists:
            trackers[next_id] = {
                'centroid': f['centroid'],
                'bbox': f['bbox'],
                'kps': f['kps'],
                'name': f['name'],
                'emotion_hist': deque(maxlen=EMOTION_WINDOW),
                'prev_kpts': None,
                'missing': 0,
                'kpts_yolo': None
            }
            active_ids.append(next_id)
            next_id += 1

    for tid in active_ids:
        best_sk, min_d = None, 150
        for sk in skeletons:
            d = dist.euclidean(sk['nose'], trackers[tid]['centroid'])
            if d < min_d:
                min_d, best_sk = d, sk['kpts']
        trackers[tid]['kpts_yolo'] = best_sk

    cam_dx, cam_dy = get_camera_shift(trackers)

    # 4. HANDSHAKE YOLO
    handshake_pairs_yolo = []
    ids_body = [tid for tid in active_ids if trackers[tid]['kpts_yolo'] is not None]
    for i in range(len(ids_body)):
        for j in range(i + 1, len(ids_body)):
            id1, id2 = ids_body[i], ids_body[j]
            if is_valid_handshake_vector(trackers[id1]['kpts_yolo'], trackers[id2]['kpts_yolo']):
                handshake_pairs_yolo.extend([id1, id2])

    # 5. PROCESSAMENTO POR PESSOA
    for tid in list(trackers.keys()):
        if tid not in active_ids:
            trackers[tid]['missing'] += 1
            if trackers[tid]['missing'] > 10:
                del trackers[tid]
            continue

        data = trackers[tid]
        x1, y1, x2, y2 = data['bbox']
        safe_x1, safe_y1, safe_x2, safe_y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        if safe_x2 <= safe_x1:
            continue

        angle = get_face_rotation(data.get('kps'))
        try:
            if angle < 35:
                face_img = frame[safe_y1:safe_y2, safe_x1:safe_x2]
                res = DeepFace.analyze(face_img, actions=['emotion'], enforce_detection=False, silent=True)
                data['emotion_hist'].append(clean_emotion(res[0]['dominant_emotion'], res[0]['emotion'][res[0]['dominant_emotion']], angle))
            else:
                data['emotion_hist'].append("neutral")
        except:
            pass

        activity, is_anomaly = "Parado", False
        if data['kpts_yolo'] is not None:
            activity, is_anomaly = analyze_motion_compensated(data['kpts_yolo'], data['prev_kpts'], cam_dx, cam_dy)
            data['prev_kpts'] = data['kpts_yolo']
            for p1, p2 in [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (11, 12)]:
                if data['kpts_yolo'][p1][2] > 0.5 and data['kpts_yolo'][p2][2] > 0.5:
                    cv2.line(frame, (int(data['kpts_yolo'][p1][0]), int(data['kpts_yolo'][p1][1])),
                             (int(data['kpts_yolo'][p2][0]), int(data['kpts_yolo'][p2][1])), (0, 255, 255), 2)

        # Sobrescreve se for handshake
        if tid in handshake_pairs_yolo:
            activity = "Aperto de Mao"

        global_stats['activities'].append(activity)
        if is_anomaly:
            global_stats['anomalies'] += 1

        emo = Counter(data['emotion_hist']).most_common(1)[0][0] if data['emotion_hist'] else "..."
        global_stats['emotions'].append(emo)

        color = (255, 0, 255) if activity == "Aperto de Mao" else ((0, 0, 255) if is_anomaly else (0, 255, 0))
        cv2.rectangle(frame, (safe_x1, safe_y1), (safe_x2, safe_y2), color, 2)
        label = f"{data['name']} | {emo.upper()} | {activity}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.putText(frame, label, (max(0, min(safe_x1, w - tw)), safe_y1 - 10 if safe_y1 - 10 > th else safe_y2 + th + 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # 6. DRAW MEDIAPIPE CLOSE-UP BOX
    if mp_handshake_box:
        cx, cy = mp_handshake_box
        cv2.rectangle(frame, (cx - 100, cy - 60), (cx + 100, cy + 60), (255, 0, 255), 4)
        cv2.putText(frame, "HANDSHAKE (CLOSE-UP)", (cx - 100, cy - 70), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)

    out.write(frame)
    if global_stats['frames'] % LOG_FRAMES == 0:
        print(f"Frame {global_stats['frames']}...")


cap.release()
out.release()
generate_report()
print(f"✅ VÍDEO PRONTO: {OUTPUT_VIDEO}")


⚙️ Inicializando Modelos...
✅ Tudo pronto.
▶️ Processando: /content/drive/MyDrive/Colab Notebooks/tech 04/video/input_video.mp4


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5


26-01-08 12:43:41 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


100%|██████████| 5.98M/5.98M [00:00<00:00, 329MB/s]


Frame 50...
Frame 100...
Frame 150...
Frame 200...
Frame 250...
Frame 300...
Frame 350...
Frame 400...
Frame 450...
Frame 500...
Frame 550...
Frame 600...
Frame 650...
Frame 700...
Frame 750...
Frame 800...
Frame 850...
Frame 900...
Frame 950...
Frame 1000...
Frame 1050...
Frame 1100...
Frame 1150...
Frame 1200...
Frame 1250...
Frame 1300...
Frame 1350...
Frame 1400...
Frame 1450...
Frame 1500...
Frame 1550...
Frame 1600...
Frame 1650...
Frame 1700...
Frame 1750...
Frame 1800...
Frame 1850...
Frame 1900...
Frame 1950...
Frame 2000...
Frame 2050...
Frame 2100...
Frame 2150...
Frame 2200...
Frame 2250...
Frame 2300...
Frame 2350...
Frame 2400...
Frame 2450...
Frame 2500...
Frame 2550...
Frame 2600...
Frame 2650...
Frame 2700...
Frame 2750...
Frame 2800...
Frame 2850...
Frame 2900...
Frame 2950...
Frame 3000...
Frame 3050...
Frame 3100...
Frame 3150...
Frame 3200...
Frame 3250...
Frame 3300...

📊 RELATÓRIO FINAL V18
Frames: 3326 | Anomalias: 19
--------------------------------------------